# 03 · Figures

**How to use:** *Runtime → Run all*. Nothing to edit. Allow Google Drive access when asked.

If Colab disconnects or the runtime ends, open the notebook again and *Run all* again: finished
work is kept on Drive and skipped, and the run continues where it stopped. All outputs,
checkpoints and caches are written to Drive (never to the temporary Colab disk).

Builds the publication figures **only from saved outputs** (graph run from notebook 01, experiment
results from notebook 02). No training, no EEG processing. Figures whose inputs do not exist yet are
skipped and listed. Each run writes a new folder `figures/figures_<time>/` with the figures, the
plotted data and a manifest; nothing is overwritten. See `docs/figures.md`.

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_CODE = False   # leave False. True only if you were asked to switch to newer code
                      # (all notebooks then use the newest commit; unfinished work is recomputed).
BRANCH = 'claude/ecstatic-davinci-a2e8mh'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
print('outputs on Drive:', OUT)

## Generate figures

In [ ]:
from pathlib import Path
from adgraph.pipeline import latest_complete_graph_run

GRAPHS = latest_complete_graph_run(OUT)
RESULTS = f"{OUT}/experiments"
print('graph run:', GRAPHS)
sh(f"python scripts/make_figures.py --config configs/figures/paper.yaml --results '{RESULTS}' "
   f"--graphs '{GRAPHS}' --out '{OUT}/figures'")
latest = sorted(Path(f"{OUT}/figures").glob('figures_*'))[-1]
print('figures in', latest)
for f in sorted(latest.rglob('*.png')):
    print(' ', f.relative_to(latest))